# 📊 Sales Trend Visualization
## CodTech Internship — Task 1 | Data Analytics Domain

**Intern:** Yash Gamare  
**Domain:** Data Science & Analytics  
**Project:** Sales Trend Visualization (Advanced Level)

---

### Project Scope
This notebook performs end-to-end sales data analysis including:
- ✅ Data Loading & Cleaning
- ✅ Exploratory Data Analysis (EDA)
- ✅ Time-Series Sales Trend Visualization
- ✅ Category & Region Breakdown
- ✅ Revenue Heatmaps
- ✅ Sales Rep Performance Analysis
- ✅ Polynomial Regression Forecasting
- ✅ Correlation Analysis
- ✅ Executive Dashboard


## 1. Import Libraries & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Dark theme styling
plt.rcParams.update({
    'figure.facecolor': '#0f172a',
    'axes.facecolor': '#1e293b',
    'axes.edgecolor': '#334155',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'text.color': '#e2e8f0',
    'grid.color': '#334155',
    'grid.alpha': 0.5,
})

COLORS = ['#6366f1', '#22d3ee', '#f59e0b', '#10b981', '#f43f5e', '#a78bfa']
print("Libraries loaded successfully!")

## 2. Data Loading & Cleaning

In [ ]:
df = pd.read_csv('data/sales_data.csv', parse_dates=['Date'])
print(f"Shape: {df.shape}")
print(f"Date Range: {df['Date'].min().date()} → {df['Date'].max().date()}")
df.head(10)

In [ ]:
# Data Info & Missing Values
print("Data Types:")
print(df.dtypes)
print(f"\nMissing Values:\n{df.isnull().sum()}")
print(f"\nDuplicates: {df.duplicated().sum()}")

In [ ]:
# Feature Engineering
df['Month'] = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.strftime('%b')
df['Quarter'] = df['Date'].dt.quarter.map({1:'Q1',2:'Q2',3:'Q3',4:'Q4'})
df['Revenue_Lakh'] = df['Revenue'] / 100000
df['Effective_Price'] = df['Unit_Price'] * (1 - df['Discount_Pct']/100)
df['Profit_Margin'] = ((df['Effective_Price'] - df['Unit_Price'] * 0.6) / df['Effective_Price']) * 100

print("Feature Engineering Complete!")
print(f"\nRevenue Stats (₹ Lakhs):\n{df['Revenue_Lakh'].describe().round(2)}")

## 3. Monthly Revenue Trend & Polynomial Forecast

In [ ]:
monthly = df.groupby('Month').agg(
    Revenue=('Revenue_Lakh','sum'),
    Units=('Units_Sold','sum')
).reset_index()
monthly['Month_Name'] = pd.to_datetime(monthly['Month'], format='%m').dt.strftime('%b')

# Polynomial Regression
X = monthly['Month'].values.reshape(-1, 1)
y = monthly['Revenue'].values
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X)
model = LinearRegression().fit(X_poly, y)
y_pred = model.predict(X_poly)
r2 = r2_score(y, y_pred)
mae = mean_absolute_error(y, y_pred)

print(f"Model R²  : {r2:.4f}")
print(f"Model MAE : ₹{mae:.2f} Lakhs")
print(monthly[['Month_Name','Revenue','Units']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7), facecolor='#0f172a')

ax1 = axes[0]
ax1.fill_between(monthly['Month'], monthly['Revenue'], alpha=0.2, color='#6366f1')
ax1.plot(monthly['Month'], monthly['Revenue'], color='#6366f1', linewidth=2.5,
         marker='o', markersize=8, markerfacecolor='white', label='Actual Revenue')
ax1.plot(monthly['Month'], y_pred, '--', color='#f59e0b', linewidth=2,
         label=f'Poly Forecast (R²={r2:.3f})')
ax1.set_title('Revenue Trend with Polynomial Forecast', fontsize=13, color='#94a3b8')
ax1.set_xticks(monthly['Month']); ax1.set_xticklabels(monthly['Month_Name'])
ax1.legend(facecolor='#1e293b', edgecolor='#334155', labelcolor='white')
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
bars = ax2.bar(monthly['Month_Name'], monthly['Units'], color=COLORS[:12], edgecolor='white', linewidth=0.5)
ax2.set_title('Monthly Units Sold', fontsize=13, color='#94a3b8')
ax2.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Category & Region Analysis

In [ ]:
cat_rev = df.groupby('Product_Category')['Revenue_Lakh'].sum().sort_values(ascending=False)
region_rev = df.groupby('Region')['Revenue_Lakh'].sum().sort_values(ascending=False)
print("Category Revenue (₹ Lakhs):")
print(cat_rev.round(2))
print("\nRegion Revenue (₹ Lakhs):")
print(region_rev.round(2))

In [ ]:
region_cat = df.pivot_table(index='Region', columns='Product_Category',
                             values='Revenue_Lakh', aggfunc='sum', fill_value=0)
fig, axes = plt.subplots(1, 3, figsize=(20, 7), facecolor='#0f172a')

# Donut
ax1 = axes[0]
wedges, texts, autotexts = ax1.pie(cat_rev.values, labels=cat_rev.index, autopct='%1.1f%%',
    startangle=90, colors=COLORS[:len(cat_rev)], pctdistance=0.75,
    wedgeprops=dict(width=0.55, edgecolor='#0f172a', linewidth=2))
for at in autotexts: at.set_color('white'); at.set_fontsize(9)
ax1.set_title('Revenue by Category', color='#94a3b8')
ax1.text(0, 0, f'₹{cat_rev.sum():.0f}L', ha='center', va='center', fontsize=12, color='white', fontweight='bold')

# Region bar
ax2 = axes[1]
ax2.barh(region_rev.index, region_rev.values, color=COLORS[:4], edgecolor='white')
ax2.set_title('Revenue by Region', color='#94a3b8')
ax2.grid(True, axis='x', alpha=0.3)

# Stacked
region_cat.plot(kind='bar', stacked=True, ax=axes[2], color=COLORS[:5], edgecolor='white')
axes[2].set_title('Region × Category Stacked', color='#94a3b8')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=0)
axes[2].grid(True, axis='y', alpha=0.3)

plt.tight_layout(); plt.show()

## 5. Revenue Heatmap

In [ ]:
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
pivot = df.pivot_table(index='Product_Category', columns='Month_Name',
                        values='Revenue_Lakh', aggfunc='sum', fill_value=0)
pivot = pivot.reindex(columns=[m for m in month_order if m in pivot.columns])

fig, ax = plt.subplots(figsize=(16, 5), facecolor='#0f172a')
sns.heatmap(pivot, ax=ax, cmap='YlOrRd', annot=True, fmt='.0f',
            linewidths=0.5, linecolor='#0f172a', annot_kws={'size': 9, 'color': 'black'})
ax.set_title('Revenue Heatmap: Category × Month (₹ Lakhs)', fontsize=14, color='white', pad=12)
plt.tight_layout(); plt.show()

## 6. Sales Rep Performance & Customer Segment

In [ ]:
rep_perf = df.groupby('Sales_Rep').agg(
    Revenue=('Revenue_Lakh','sum'),
    Units=('Units_Sold','sum'),
    Deals=('Date','count')
).sort_values('Revenue', ascending=False)
print(rep_perf.round(2))

In [ ]:
seg_rev = df.groupby('Customer_Segment')['Revenue_Lakh'].sum()
fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor='#0f172a')

x = np.arange(len(rep_perf))
axes[0].bar(x - 0.175, rep_perf['Revenue'], 0.35, label='Revenue (₹L)', color=COLORS[0], edgecolor='white')
ax_r = axes[0].twinx()
ax_r.bar(x + 0.175, rep_perf['Deals'], 0.35, label='Deals', color=COLORS[1], edgecolor='white')
axes[0].set_xticks(x); axes[0].set_xticklabels(rep_perf.index, rotation=15)
axes[0].set_title('Sales Rep Performance', color='#94a3b8')

axes[1].pie(seg_rev.values, labels=seg_rev.index, autopct='%1.1f%%', startangle=140,
    colors=COLORS[:3], pctdistance=0.75, wedgeprops=dict(width=0.55))
axes[1].set_title('Revenue by Customer Segment', color='#94a3b8')

plt.tight_layout(); plt.show()

## 7. Quarterly Analysis

In [ ]:
quarterly = df.groupby('Quarter').agg(
    Revenue=('Revenue_Lakh','sum'),
    Units=('Units_Sold','sum')
).reset_index()
print(quarterly)
print("\nQuarter-over-Quarter Growth:")
for i in range(1, len(quarterly)):
    g = ((quarterly['Revenue'].iloc[i] - quarterly['Revenue'].iloc[i-1]) / quarterly['Revenue'].iloc[i-1]) * 100
    print(f"  {quarterly['Quarter'].iloc[i-1]} → {quarterly['Quarter'].iloc[i]}: {g:+.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor='#0f172a')
axes[0].bar(quarterly['Quarter'], quarterly['Revenue'], color=COLORS[:4], edgecolor='white', width=0.5)
axes[0].plot(quarterly['Quarter'], quarterly['Revenue'], 'o--', color='#f59e0b', linewidth=2)
axes[0].set_title('Quarterly Revenue', color='#94a3b8')
axes[0].grid(True, axis='y', alpha=0.3)

tp = df.groupby('Product_Name')['Revenue_Lakh'].sum().sort_values(ascending=True)
axes[1].barh(tp.index, tp.values, color=plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(tp))), edgecolor='white')
axes[1].set_title('Revenue by Product', color='#94a3b8')
axes[1].grid(True, axis='x', alpha=0.3)

plt.tight_layout(); plt.show()

## 8. Correlation Analysis

In [ ]:
corr_cols = ['Units_Sold', 'Unit_Price', 'Revenue_Lakh', 'Discount_Pct', 'Profit_Margin']
corr = df[corr_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor='#0f172a')
sns.heatmap(corr, ax=axes[0], annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
axes[0].set_title('Correlation Matrix', color='#94a3b8')

for cat in df['Product_Category'].unique():
    mask = df['Product_Category'] == cat
    axes[1].scatter(df[mask]['Units_Sold'], df[mask]['Revenue_Lakh'], label=cat, alpha=0.7, s=60)
z = np.polyfit(df['Units_Sold'], df['Revenue_Lakh'], 1)
x_line = np.linspace(df['Units_Sold'].min(), df['Units_Sold'].max(), 100)
axes[1].plot(x_line, np.poly1d(z)(x_line), '--', color='#f59e0b', linewidth=2)
axes[1].legend(facecolor='#1e293b', labelcolor='white')
axes[1].set_title('Units Sold vs Revenue', color='#94a3b8')
axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 9. Key Insights & Conclusions

---

### 📈 Revenue Performance
- **Total Revenue:** ₹1,501 Lakhs across FY 2023
- Strong **Q3 → Q4** growth of **+24.9%**, driven by year-end demand
- **December** was the peak sales month

### 🏆 Top Performers
- **Best Region:** North (highest revenue contribution)
- **Best Category:** Electronics (dominant share)
- **Best Product:** Smartphone Z (highest individual revenue)
- **Best Sales Rep:** Ravi Kumar

### 📊 Forecast Accuracy
- Polynomial Regression model achieved **R² = 0.9334**, indicating strong predictive performance

### 💡 Business Recommendations
1. Increase **Electronics inventory** heading into Q4
2. Expand **North region** strategies to **East & South**
3. Run **seasonal promotions** in Q1–Q2 to smooth revenue dips
4. Train underperforming reps using Ravi Kumar's best practices
